
# Task 2 — Establish the Judge's Smallest Score Difference

## Objective

Measure how much the LLM judge's scores vary when the same translations are evaluated repeatedly without changing the source, reference, candidate, or evaluation prompt.

The goal is to estimate the minimum score difference that can be treated as meaningful rather than normal evaluation variability.

## Experimental Plan

- Use the team's committed judge configuration.
- Evaluate a fixed set of English-to-Mandarin translations.
- Run each evaluation three times at temperature 0.
- Repeat three times at the configured temperature.
- Calculate score ranges per example and their average.
- Estimate the standard error of the mean at the current evaluation set size.
- Determine a minimum actionable score difference and required sample size.
- Record model configuration, machine details, and library versions.

## Deliverables

1. Reproducible experiment notebook and results file.
2. Short written findings with the recommended decision threshold.



In [1]:

import os
import sys
import json
import hashlib
import platform
import numpy as np
import pandas as pd
import torch
import transformers

from datetime import datetime, timezone

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

environment = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "pytorch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    )
}

print(json.dumps(environment, indent=2))


{
  "timestamp_utc": "2026-10-22T01:52:17.470844+00:00",
  "python_version": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "machine": "x86_64",
  "pytorch_version": "2.11.0+cu130",
  "transformers_version": "5.20.1",
  "numpy_version": "2.1.3",
  "pandas_version": "2.2.3",
  "cuda_available": true,
  "gpu_name": "Tesla T4"
}


In [2]:


eval_df = pd.read_csv(
    "wmt17_en_zh_judge_eval_50.csv"
)

required_columns = [
    "eval_id",
    "source_en",
    "reference_zh",
    "candidate_zh"
]

assert all(
    col in eval_df.columns
    for col in required_columns
), "Missing required dataset columns."

assert eval_df["eval_id"].nunique() == 50

print("Dataset loaded successfully.")
print("Examples:", len(eval_df))

display(eval_df.head())


Dataset loaded successfully.
Examples: 50


,eval_id,source_en,reference_zh,candidate_zh
0,E001,Democratic members of the House Intelligence C...,众议院情报委员会的民主党成员于周四公布了其调查结果，与共和党专项工作小组的部分结论相一致。,"众议院情报委员会的民主成员在星期四公布了他们自己的调查结果,他们同意共和党工作队的一些结论。"
1,E002,"Russian gymnast Aliya Mustafina, with her larg...",长着一双会说话的大眼睛的俄罗斯体操选手穆斯塔芬娜 (Aliya Mustafina) 被指长...,"俄罗斯体育馆Aliya Mustafina, 以她大眼睛的表情, 据说是古巴裔美国人歌手Gl..."
2,E003,Gee and Schwartzman are scheduled to discuss a...,吉和斯瓦茨曼将讨论并分享1926年贝特伦·古德西的设计图片以及1933年哈迪・霍尔兹曼・弗夫...,Gee和Schwartzman计划讨论和分享Bertram Goodhue 1926年设计以...
3,E004,"Over the past decade, the State Council has is...",近十年来，国务院前后共发布过5次淘汰过剩与落后产能的通知。,"过去十年来,国务院共发出五份消除过度和落后生产能力的通知。"
4,E005,Helen Glover and Heather Stanning inspired to ...,海伦·格拉芙和希瑟·斯坦宁受身患癌症的教练的鼓舞，夺得赛艇金牌,Helen Glover和Heather Stanning在教练的癌症战役中 获得了划划船胜...


In [3]:

# Reuse the judge configuration from Task 1

JUDGE_CONFIG = {
    "model": "Qwen/Qwen3-4B",
    "prompt_version": "P2_anchored",
    "score_min": 1,
    "score_max": 5,
    "temperatures": [0.0, 0.7],
    "top_p": 0.95,
    "runs_per_temperature": 3,
    "max_new_tokens": 128,
    "seed": 42
}

JUDGE_PROMPT = """
You are evaluating the quality of an English-to-Mandarin Chinese translation.

English source:
{source}

Reference Mandarin translation:
{reference}

Candidate Mandarin translation:
{candidate}

Evaluate the candidate translation using the following criteria:

Accuracy:
Does the candidate correctly preserve the meaning of the English source?

Completeness:
Does the candidate preserve all important information without omissions
or unsupported additions?

Fluency:
Is the Mandarin grammatically correct, natural, and readable?

Terminology:
Are words, expressions, names, and domain-specific terms translated
appropriately?

Based on these criteria, give one overall score from 1 to 5 using the
following scoring anchors:

5 = Excellent
The meaning is fully preserved. There are no meaningful mistranslations,
omissions, or unsupported additions.

4 = Good
The meaning is substantially preserved. Minor issues may exist but do not
materially change the source meaning.

3 = Acceptable
The main meaning is preserved, but noticeable errors reduce translation quality.

2 = Poor
Important information is mistranslated, omitted, or incorrectly added.

1 = Very poor
The translation substantially fails to convey the source meaning.

Return your answer exactly in this format:

Score: <1-5>
Explanation: <brief explanation>
""".strip()

# Generate a reproducible configuration fingerprint
fingerprint_config = {
    **JUDGE_CONFIG,
    "prompt": JUDGE_PROMPT,
    "chat_template_thinking": False
}

config_json = json.dumps(
    fingerprint_config,
    sort_keys=True,
    ensure_ascii=False
)

CONFIG_FINGERPRINT = hashlib.sha256(
    config_json.encode("utf-8")
).hexdigest()

print("Judge model:", JUDGE_CONFIG["model"])
print("Prompt:", JUDGE_CONFIG["prompt_version"])
print("Temperatures:", JUDGE_CONFIG["temperatures"])
print("Runs per temperature:", JUDGE_CONFIG["runs_per_temperature"])
print("Score scale: 1–5")
print("Config SHA-256:", CONFIG_FINGERPRINT)


Judge model: Qwen/Qwen3-4B
Prompt: P2_anchored
Temperatures: [0.0, 0.7]
Runs per temperature: 3
Score scale: 1–5
Config SHA-256: c5a05620d78a85a4c626f72cdd732749e334d1858660db5c7f321c4ae1860dbf


In [4]:

experiment_rows = []

for _, example in eval_df.iterrows():
    for temperature in JUDGE_CONFIG["temperatures"]:
        for run in range(1, JUDGE_CONFIG["runs_per_temperature"] + 1):

            prompt = JUDGE_PROMPT.format(
                source=example["source_en"],
                reference=example["reference_zh"],
                candidate=example["candidate_zh"]
            )

            experiment_rows.append({
                "eval_id": example["eval_id"],
                "temperature": temperature,
                "run": run,
                "prompt": prompt,
                "source_en": example["source_en"],
                "reference_zh": example["reference_zh"],
                "candidate_zh": example["candidate_zh"]
            })

experiment_plan = pd.DataFrame(experiment_rows)

assert len(experiment_plan) == 300

print("Examples:", eval_df["eval_id"].nunique())
print("Temperatures:", JUDGE_CONFIG["temperatures"])
print("Runs per temperature:", JUDGE_CONFIG["runs_per_temperature"])
print("Total evaluations:", len(experiment_plan))

display(
    experiment_plan[
        ["eval_id", "temperature", "run"]
    ].head(12)
)


Examples: 50
Temperatures: [0.0, 0.7]
Runs per temperature: 3
Total evaluations: 300


,eval_id,temperature,run
0,E001,0.0,1
1,E001,0.0,2
2,E001,0.0,3
3,E001,0.7,1
4,E001,0.7,2
5,E001,0.7,3
6,E002,0.0,1
7,E002,0.0,2
8,E002,0.0,3
9,E002,0.7,1


In [5]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import re
import gc

assert torch.cuda.is_available(), "Enable GPU in Colab first."

MODEL_NAME = JUDGE_CONFIG["model"]
RESULTS_FILE = "judge_score_variability_results.csv"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()

# Record the actual GPU environment
environment["gpu_name"] = torch.cuda.get_device_name(0)
environment["cuda_available"] = True
environment["cuda_version"] = torch.version.cuda
environment["model_revision"] = getattr(
    model.config, "_commit_hash", None
)
environment["tokenizer_revision"] = getattr(
    tokenizer, "init_kwargs", {}
).get("_commit_hash")

print("GPU:", environment["gpu_name"])
print("Model revision:", environment["model_revision"])
print("Config fingerprint:", CONFIG_FINGERPRINT)


def judge_translation(prompt, temperature):

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    generation_args = {
        "max_new_tokens": JUDGE_CONFIG["max_new_tokens"],
        "pad_token_id": tokenizer.eos_token_id
    }

    if temperature == 0:
        generation_args["do_sample"] = False
    else:
        generation_args.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": JUDGE_CONFIG["top_p"]
        })

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            **generation_args
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    match = re.search(
        r"\bScore\s*:\s*([1-5])\b",
        response,
        flags=re.IGNORECASE
    )

    score = int(match.group(1)) if match else np.nan

    return score, response


# Resume previously saved results if present
if os.path.exists(RESULTS_FILE):
    results = pd.read_csv(RESULTS_FILE).to_dict("records")
else:
    results = []

completed = {
    (str(r["eval_id"]), round(float(r["temperature"]), 2), int(r["run"]))
    for r in results
}

for _, row in tqdm(
    experiment_plan.iterrows(),
    total=len(experiment_plan)
):

    key = (
        str(row["eval_id"]),
        round(float(row["temperature"]), 2),
        int(row["run"])
    )

    if key in completed:
        continue

    score, response = judge_translation(
        row["prompt"],
        float(row["temperature"])
    )

    results.append({
        "eval_id": row["eval_id"],
        "temperature": float(row["temperature"]),
        "run": int(row["run"]),
        "score": score,
        "raw_response": response,
        "config_fingerprint": CONFIG_FINGERPRINT
    })

    completed.add(key)

    if len(results) % 10 == 0:
        pd.DataFrame(results).to_csv(
            RESULTS_FILE, index=False
        )

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_FILE, index=False)

with open("judge_environment.json", "w") as f:
    json.dump(environment, f, indent=2)

print("\nExperiment finished!")
print("Total results:", len(results_df))
print("Valid scores:", results_df["score"].notna().sum())
print("Invalid scores:", results_df["score"].isna().sum())

display(
    results_df.groupby("temperature")["score"].agg(
        ["count", "mean", "std"]
    )
)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

GPU: Tesla T4
Model revision: None
Config fingerprint: c5a05620d78a85a4c626f72cdd732749e334d1858660db5c7f321c4ae1860dbf


  0%|          | 0/300 [00:00<?, ?it/s]


Experiment finished!
Total results: 300
Valid scores: 300
Invalid scores: 0


,count,mean,std
temperature,,,
0.0,150,3.760000,0.428514
0.7,150,3.753333,0.432515


In [6]:

from scipy import stats
import math

# Check that every example has 3 valid scores
assert results_df["score"].notna().all()
assert len(results_df) == 300

counts = results_df.groupby(
    ["eval_id", "temperature"]
)["score"].count()

assert (counts == 3).all(), "Some examples do not have 3 runs."

# Per-example variability
summary = results_df.groupby(
    ["eval_id", "temperature"]
)["score"].agg(
    mean_score="mean",
    min_score="min",
    max_score="max",
    score_sd="std"
).reset_index()

summary["score_range"] = (
    summary["max_score"] - summary["min_score"]
)

temperature_summary = summary.groupby(
    "temperature"
).agg(
    examples=("eval_id", "count"),
    average_range=("score_range", "mean"),
    maximum_range=("score_range", "max"),
    average_within_sd=("score_sd", "mean"),
    changed_examples=("score_range", lambda x: (x > 0).sum())
).reset_index()

print("WITHIN-EXAMPLE VARIABILITY")
display(temperature_summary)

# Estimate measurement noise from within-example variances
# For an average of 3 runs over 50 examples
n = summary["eval_id"].nunique()
r = 3

noise_results = []

for temp in [0.0, 0.7]:
    subset = summary[summary["temperature"] == temp]

    within_variance = (subset["score_sd"] ** 2).mean()
    noise_sd = math.sqrt(within_variance)

    # SE from repeated-judgment noise only
    measurement_se = noise_sd / math.sqrt(n * r)

    # Approximate 95% threshold for comparing
    # two independently repeated set means
    min_difference = 1.96 * math.sqrt(2) * measurement_se

    # Full SE across examples, including example differences
    set_se = subset["mean_score"].std(ddof=1) / math.sqrt(n)

    noise_results.append({
        "temperature": temp,
        "within_noise_sd": noise_sd,
        "measurement_se": measurement_se,
        "set_mean_se": set_se,
        "minimum_difference_95": min_difference
    })

noise_df = pd.DataFrame(noise_results)

print("\nMEASUREMENT UNCERTAINTY")
display(noise_df.round(4))

# Compare matched examples across temperatures
paired = summary.pivot(
    index="eval_id",
    columns="temperature",
    values="mean_score"
)

paired_difference = paired[0.7] - paired[0.0]

print("\nTEMPERATURE COMPARISON")
print("Mean paired difference:", round(paired_difference.mean(), 4))
print("Examples with different mean scores:",
      int((paired_difference != 0).sum()), "/", n)

# Required set size for a target score difference
# Approximate two-sided 5% significance, 80% power
# Based only on repeat-judgment noise
TARGET_DIFFERENCE = 0.2

sigma = noise_df.loc[
    noise_df["temperature"] == 0.7,
    "within_noise_sd"
].iloc[0]

required_n = math.ceil(
    2 * (1.96 + 0.84)**2 * sigma**2 /
    (r * TARGET_DIFFERENCE**2)
)

print("\nSAMPLE SIZE ESTIMATE")
print("Target difference:", TARGET_DIFFERENCE)
print("Estimated examples required:", max(1, required_n))
print("Current examples:", n)

# Save analysis outputs
summary.to_csv("judge_per_example_variability.csv", index=False)
temperature_summary.to_csv("judge_temperature_summary.csv", index=False)
noise_df.to_csv("judge_measurement_uncertainty.csv", index=False)

print("\nAnalysis files saved.")


WITHIN-EXAMPLE VARIABILITY


,temperature,examples,average_range,maximum_range,average_within_sd,changed_examples
0,0.0,50,0.00,0,0.000000,0
1,0.7,50,0.06,1,0.034641,3



MEASUREMENT UNCERTAINTY


,temperature,within_noise_sd,measurement_se,set_mean_se,minimum_difference_95
0,0.0,0.0000,0.0000,0.0610,0.000
1,0.7,0.1414,0.0115,0.0593,0.032



TEMPERATURE COMPARISON
Mean paired difference: -0.0067
Examples with different mean scores: 3 / 50

SAMPLE SIZE ESTIMATE
Target difference: 0.2
Estimated examples required: 3
Current examples: 50

Analysis files saved.


In [7]:

import numpy as np
import pandas as pd
from scipy.stats import norm

# Use temperature 0.7 as the operational setting
subset = summary[summary["temperature"] == 0.7].copy()

n = len(subset)
runs = 3

# Observed variance of per-example mean scores
variance_means = subset["mean_score"].var(ddof=1)

# Within-example judgment noise
within_variance = (subset["score_sd"] ** 2).mean()

# Estimate between-example variance
between_variance = max(
    0,
    variance_means - within_variance / runs
)

# Total variance of a 3-run example mean
total_variance = between_variance + within_variance / runs

# Standard error of one set mean
set_se = np.sqrt(total_variance / n)

# Conservative threshold for two independent set means
minimum_actionable = 1.96 * np.sqrt(2) * set_se

# Set size required to detect a chosen difference
# 5% two-sided significance, 80% power
target_difference = 0.2

z_alpha = norm.ppf(0.975)
z_power = norm.ppf(0.80)

required_n = np.ceil(
    2 * (z_alpha + z_power)**2
    * total_variance
    / target_difference**2
)

print("FINAL MEASUREMENT SUMMARY")
print("-------------------------")
print("Examples:", n)
print("Runs per example:", runs)
print("Temperature: 0.7")
print("Within-example variance:", round(within_variance, 5))
print("Between-example variance:", round(between_variance, 5))
print("Standard error of set mean:", round(set_se, 4))
print("95% comparison threshold:",
      round(minimum_actionable, 3))
print("Recommended actionable difference:",
      round(np.ceil(minimum_actionable * 100) / 100, 2))
print("Examples per independent set to detect 0.2:",
      int(required_n))


FINAL MEASUREMENT SUMMARY
-------------------------
Examples: 50
Runs per example: 3
Temperature: 0.7
Within-example variance: 0.02
Between-example variance: 0.16934
Standard error of set mean: 0.0593
95% comparison threshold: 0.164
Recommended actionable difference: 0.17
Examples per independent set to detect 0.2: 70
